# Monitoring a Slurm HPC — hands-on

Run this notebook **on a Slurm login node** (sinfo / squeue / scontrol on
PATH). Every cell runs one monitoring script against your live cluster and
prints what it found — run them top to bottom. The command reference behind
every call is [SLURM_TUTORIAL.md](../../SLURM_TUTORIAL.md) at the repo root.

| Script | What it shows | Slurm commands it runs |
|---|---|---|
| `hpc_info.py` | partitions, CPUs, GPUs, memory, time limits; your access | `sinfo`, `sacctmgr` |
| `show_free_nodes.py` | nodes with free CPUs / GPUs, ready-to-paste `srun` commands | `sinfo` |
| `scontroller_nodes.py` | per-node detail (State, TRES, features) + slurm.conf resource keys | `scontrol show node` |
| `slurm_job_checker.py` | recent jobs of given users — failed / stuck / unkillable | `sacct`, `scontrol show job` |
| `hpc_dashboard.py` | the live dashboard — see the last cell for deployment | all of the above |


## 0. No cluster at hand? Use the bundled fake Slurm

This folder ships `fake_slurm/` — a small simulated cluster: the fictional
**kapoorlabs-fr** organization (France), whose GPU nodes are named after
their GPUs — `h100-*`, `h200-*`, `b200-*` — plus two `cpu-*` nodes. Run the
cell below and every command in this notebook answers from that cluster.
**On a real login node, skip this cell.**

In [1]:
import os
from pathlib import Path

fake_slurm = Path("fake_slurm").resolve()
os.environ["PATH"] = f"{fake_slurm}{os.pathsep}{os.environ['PATH']}"
os.environ["SLURM_CONF"] = str(fake_slurm / "slurm.conf")
print("Slurm commands now resolve to the bundled fake_slurm/ (kapoorlabs-fr)")
!sinfo -h

Slurm commands now resolve to the bundled fake_slurm/ (kapoorlabs-fr)
slurm 23.02.4


## 1. Cluster inventory — `hpc_info.py`

One table of every partition (nodes, CPUs, GPU types, memory, time limit),
cluster-wide GPU totals over unique nodes, and your own account / partition
/ QOS access from `sacctmgr`.

In [2]:
!python hpc_info.py

  PARTITIONS
PARTITION  NODES  CPUS  GPUs            MEM(GB)  TIMELIMIT  
---------  -----  ----  --------------  -------  -----------
b200       2      128   16× B200.192GB  4000     14-00:00:00
cpu        2      256   -               1000     3-00:00:00 
h100       4      128   32× H100.80GB   2000     7-00:00:00 
h200       3      144   24× H200.141GB  3000     7-00:00:00 

Cluster GPU totals (unique nodes): 72 GPUs
    32× H100.80GB
    24× H200.141GB
    16× B200.192GB

Your Slurm access (user=vkapoor):
  account=kapoorlabs   partition=ALL            qos=normal


## 2. Free nodes right now — `show_free_nodes.py`

Nodes with idle CPUs, split into GPU and CPU-only, sorted by idle CPUs —
and a ready-to-paste `srun` command for the best free node of each kind.

In [3]:
!python show_free_nodes.py


=== GPU NODES ===

Node         Idle/Total CPUs  GPU                  Memory     State     
----------------------------------------------------------------------
b200-2         64/64         b200.192gb x8        4096GB      drained   
h200-3         48/48         h200.141gb x8        3072GB      idle      
h100-4         32/32         h100.80gb x8         2048GB      idle      
b200-1         32/64         b200.192gb x8        4096GB      mixed     
h200-1         24/48         h200.141gb x8        3072GB      mixed     
h100-1         16/32         h100.80gb x8         2048GB      mixed     

=== CPU-ONLY NODES ===

Node         Idle/Total CPUs  GPU                  Memory     State     
----------------------------------------------------------------------
cpu-2         128/128        -                    1024GB      idle      
cpu-1          64/128        -                    1024GB      mixed     

--- Quick commands ---
GPU: srun --nodelist=h100-1 --gres=gpu:1 --pty bash
CPU: sr

## 3. Node detail and cluster config — `scontroller_nodes.py`

`scontrol show node` per node: State, CfgTRES vs AllocTRES (configured vs
promised), GRES, features — plus the resource keys from `slurm.conf`
(memory defaults and ceilings, accounting configuration).

In [4]:
!python scontroller_nodes.py

=== SLURM Cluster Node Status (11 nodes) ===

Node: h100-1
  ActiveFeatures: gpu,large-mem
  AvailableFeatures: gpu,large-mem,x86_64
  Gres: gpu:h100.80gb:8(S:0-1)
  State: mixed
  Partitions: h100
  AllocTRES: cpu=16,mem=512G,gres/gpu=8
  CfgTRES: cpu/32,mem/2048000M,billing/32
  mem: 2048000

Node: h100-2
  ActiveFeatures: gpu,large-mem
  AvailableFeatures: gpu,large-mem,x86_64
  Gres: gpu:h100.80gb:8(S:0-1)
  State: allocated*
  Partitions: h100
  AllocTRES: cpu=32,mem=768G,gres/gpu=8
  CfgTRES: cpu/32,mem/2048000M,billing/32
  mem: 2048000

Node: h100-3
  ActiveFeatures: gpu,large-mem
  AvailableFeatures: gpu,large-mem,x86_64
  Gres: gpu:h100.80gb:8(S:0-1)
  State: allocated
  Partitions: h100
  AllocTRES: cpu=32,mem=1T,gres/gpu=8
  CfgTRES: cpu/32,mem/2048000M,billing/32
  mem: 2048000

Node: h100-4
  ActiveFeatures: gpu,large-mem
  AvailableFeatures: gpu,large-mem,x86_64
  Gres: gpu:h100.80gb:8(S:0-1)
  State: idle
  Partitions: h100
  AllocTRES: (none)
  CfgTRES: cpu/32,mem/2048

## 4. Check users' jobs — `slurm_job_checker.py`

Every job of the given users in the lookback window (users, hours and output live in `conf/job_checker.yaml` — override on the command line, e.g. `users=[varun] hours=48`), with State and
ExitCode from `sacct` and full detail from `scontrol show job`. Flags
failed, stuck and unkillable jobs. (It exits non-zero when it finds
problems — that is the signal, not a crash.)

In [5]:
!python slurm_job_checker.py 'users=[varun]' hours=24 || echo "(exit code 1 — issues flagged above)"

SLURM Job Status Checker
Users: varun
Time range: Last 24 hours

Checking user: varun
Found 5 jobs in last 24 hours
  [1/5] Checking job 10020_74... [RUNNING]
  [2/5] Checking job 3003... [RUNNING]
  [3/5] Checking job 9001... [COMPLETED]
  [4/5] Checking job 9002... [FAILED]
  [5/5] Checking job 9008... [COMPLETED]

SUMMARY
Total jobs checked: 5
  Running:      2
  Failed:       1
  Unkillable:   0
  Problematic:  0
  Completed:    2

USER: varun

FAILED JOBS (1):
----------------------------------------------------------------------

  Job ID: 9002
    Name:       pretrain
    State:      FAILED
    Exit Code:  1:0
    Node:       h100-1
    Start:      2026-11-18T09:00:00
    End:        2026-11-19T09:00:00
    Reason:     N/A

RUNNING JOBS (2):
----------------------------------------------------------------------

  Job ID: 10020_74
    Name:       pretrain
    State:      RUNNING
    Exit Code:  0:0
    Node:       h100-[1-2]
    Start:      2026-11-22T08:00:00
    End:        Un

## 9. The live dashboard — `hpc_dashboard.py`

Everything above, live: three tabs (partitions, node detail, top jobs)
with the cluster's real-time state — nothing hardcoded, it works on any
Slurm cluster. There is no tutorial for building it; it is the finished
tool. To **deploy** it on your head node:

```bash
tmux new -s dashboard                 # survive a dropped connection
streamlit run hpc_dashboard.py       # starts on port 8501
```

Then, **from your local machine**, tunnel the port:

```bash
ssh -N -L 8501:localhost:8501 <user>@<headnode>
```

and open **http://localhost:8501** in your local browser to watch the HPC
performance live.

Note: auto-refresh does not travel through the tunnel — the page fetches
new data when you interact with it or click *Refresh now*, not on a
timer.